In [0]:
import tarfile
import os
BASE_PATH = "/Volumes/persona-path/default/review_data/"

business_df = spark.read.json(BASE_PATH + "yelp_academic_dataset_business.json")
review_df   = spark.read.json(BASE_PATH + "yelp_academic_dataset_review.json")
user_df     = spark.read.json(BASE_PATH + "yelp_academic_dataset_user.json")

print("business:", business_df.count())
print("reviews :", review_df.count())
print("users   :", user_df.count())

In [0]:
business_df.createOrReplaceTempView("businesses")
review_df.createOrReplaceTempView("reviews")
user_df.createOrReplaceTempView("users")

print("All views registered.")

In [0]:
%sql
SELECT 
    city,
    state,
    COUNT(*)            AS business_count,
    SUM(review_count)   AS total_reviews,
    ROUND(AVG(stars),2) AS avg_stars
FROM businesses
GROUP BY city, state
ORDER BY business_count DESC
LIMIT 30;

In [0]:
%sql
SELECT
    city,
    ROUND(COUNT(CASE WHEN review_count < 10 THEN business_id END) / COUNT(business_id), 2) AS review_0_9_perc,
    ROUND(COUNT(CASE WHEN review_count >= 10 AND review_count < 25 THEN business_id END) / COUNT(business_id), 2)*100 AS review_10_24_perc,
    ROUND(COUNT(CASE WHEN review_count >= 25 AND review_count < 50 THEN business_id END) / COUNT(business_id), 2)*100 AS review_25_49_perc,
    ROUND(COUNT(CASE WHEN review_count >= 50 AND review_count < 100 THEN business_id END) / COUNT(business_id), 2)*100 AS review_50_99_perc,
    ROUND(COUNT(CASE WHEN review_count >= 100 AND review_count < 250 THEN business_id END) / COUNT(business_id), 2)*100 AS review_100_249_perc,
    ROUND(COUNT(CASE WHEN review_count >= 250 AND review_count < 500 THEN business_id END) / COUNT(business_id), 2)*100 AS review_250_499_perc,
    ROUND(COUNT(CASE WHEN review_count >= 500 THEN business_id END) / COUNT(business_id), 2)*100 AS review_500_perc
FROM businesses
WHERE city IN ('Philadelphia', 'New Orleans', 'Nashville', 'Tampa')
GROUP BY city;

In [0]:
%sql
SELECT
    threshold,
    COUNT(*) AS businesses_remaining
FROM businesses
CROSS JOIN (
    SELECT explode(array(10, 25, 50, 100, 200)) AS threshold
)
WHERE city = 'New Orleans'    -- swap after Cell A
  AND review_count >= threshold
GROUP BY threshold
ORDER BY threshold;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW businesses_filtered AS
SELECT *
FROM businesses
WHERE city        = 'New Orleans'   -- change to your city
  AND review_count >= 50;            -- change to your threshold

SELECT COUNT(*) AS business_count FROM businesses_filtered;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW merged AS
SELECT
    -- Review fields
    r.review_id,
    r.date              AS review_date,
    r.stars             AS review_stars,
    r.text              AS review_text,
    r.useful,
    r.funny,
    r.cool,

    -- Business fields
    b.business_id,
    b.name              AS business_name,
    b.address,
    b.city,
    b.state,
    b.postal_code,
    b.stars             AS business_stars,
    b.review_count,
    b.categories,

    -- User fields
    u.user_id,
    u.name              AS user_name,
    u.review_count      AS user_review_count,
    u.average_stars     AS user_avg_stars,
    u.fans,
    u.yelping_since

FROM reviews r
INNER JOIN businesses_filtered b ON r.business_id = b.business_id
LEFT  JOIN users u               ON r.user_id     = u.user_id;

SELECT COUNT(*) AS total_merged_rows FROM merged;

In [0]:
%sql
SELECT * FROM merged LIMIT 20;